In [1]:
import os
import json
import glob
import pandas as pd
from pathlib import Path

RAW_DIR = "C:/Users/julie/Data Eng/Raw/"
PROCESSED_DIR = "C:/Users/julie/Data Eng/Raw/Processed/"
Path(PROCESSED_DIR).mkdir(parents=True, exist_ok=True)

def load_events():
    all_files = glob.glob(os.path.join(RAW_DIR, "*.json"))
    records = []

    for file_path in all_files:
        with open(file_path, "r") as f:
            try:
                data = json.load(f)
                records.append(data)
            except json.JSONDecodeError:
                print(f"❌ Erreur JSON dans le fichier {file_path}")
    
    return pd.DataFrame(records)


def clean_events(df):
    # Convertir les timestamps
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    
    # Nettoyage des coordonnées GPS
    df["lat"] = df["location"].apply(lambda x: x.get("lat") if isinstance(x, dict) else None)
    df["lon"] = df["location"].apply(lambda x: x.get("lon") if isinstance(x, dict) else None)
    df.drop(columns=["location"], inplace=True)

    # Colonnes manquantes à remplir
    if "duration_minutes" not in df.columns:
        df["duration_minutes"] = None
    if "failure_reason" not in df.columns:
        df["failure_reason"] = None

    return df


def save_processed(df):
    df.to_csv(PROCESSED_DIR + "events_all.csv", index=False)
    for event_type in df["event"].unique():
        df[df["event"] == event_type].to_csv(
            f"{PROCESSED_DIR}events_{event_type}.csv", index=False
        )


if __name__ == "__main__":
    print("🔄 Lecture des fichiers JSON...")
    df = load_events()
    print(f"✔️ {len(df)} événements chargés.")

    print("🧹 Nettoyage...")
    df_clean = clean_events(df)

    print("💾 Sauvegarde des fichiers...")
    save_processed(df_clean)

    print("✅ Ingestion terminée. Données disponibles dans data/processed/")


🔄 Lecture des fichiers JSON...
✔️ 375 événements chargés.
🧹 Nettoyage...
💾 Sauvegarde des fichiers...
✅ Ingestion terminée. Données disponibles dans data/processed/
